## General lab instructions
For every lab in this course, you must submit **two items**:

1. A runnable Jupyter notebook with all code, figures, and outputs.
2. A formal report (PDF or Markdown) that is self-contained and answers all discussion/reflection prompts.

This notebook contains the complete implementation for **Lab 1b** plus generated outputs used in the companion report `lab1b_report.md`.


In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

np.random.seed(7)
torch.manual_seed(7)

FIG_DIR = Path("figures_lab1b")
FIG_DIR.mkdir(exist_ok=True)


## Dataset generation (Blobs, Ellipsoids, Spirals)


In [ ]:
def make_separable_blobs(n_points_per_class=120, n_classes=3, spread=0.45, seed=0):
    torch.manual_seed(seed)
    X = torch.zeros(n_points_per_class * n_classes, 2)
    y = torch.zeros(n_points_per_class * n_classes, dtype=torch.long)
    centers = []
    radius = 3.0
    for k in range(n_classes):
        theta = 2 * math.pi * k / n_classes
        centers.append(torch.tensor([radius * math.cos(theta), radius * math.sin(theta)]))
    idx = 0
    for c, center in enumerate(centers):
        X[idx:idx+n_points_per_class] = center + spread * torch.randn(n_points_per_class, 2)
        y[idx:idx+n_points_per_class] = c
        idx += n_points_per_class
    return X, y


def make_overlapping_ellipsoids(n_points_per_class=120, n_classes=3, seed=0):
    torch.manual_seed(seed)
    X = torch.zeros(n_points_per_class * n_classes, 2)
    y = torch.zeros(n_points_per_class * n_classes, dtype=torch.long)
    means = [torch.tensor([-2.0, 0.5]), torch.tensor([1.5, 1.0]), torch.tensor([0.0, -2.0])]
    cov_factors = [
        torch.tensor([[1.2, 0.6], [0.2, 0.8]]),
        torch.tensor([[0.9, -0.4], [0.4, 1.1]]),
        torch.tensor([[1.0, 0.2], [-0.5, 0.9]]),
    ]
    idx = 0
    for c in range(n_classes):
        z = torch.randn(n_points_per_class, 2)
        X[idx:idx+n_points_per_class] = z @ cov_factors[c] + means[c]
        y[idx:idx+n_points_per_class] = c
        idx += n_points_per_class
    return X, y


def make_spirals(n_points_per_class=140, n_classes=3, noise=0.18, seed=0):
    torch.manual_seed(seed)
    X = torch.zeros(n_points_per_class * n_classes, 2)
    y = torch.zeros(n_points_per_class * n_classes, dtype=torch.long)
    idx = 0
    for c in range(n_classes):
        r = torch.linspace(0.2, 4.2, n_points_per_class)
        t = torch.linspace(c * 2 * math.pi / n_classes, (c + 2.2) * 2 * math.pi / n_classes, n_points_per_class)
        t = t + noise * torch.randn(n_points_per_class)
        X[idx:idx+n_points_per_class, 0] = r * torch.sin(t)
        X[idx:idx+n_points_per_class, 1] = r * torch.cos(t)
        y[idx:idx+n_points_per_class] = c
        idx += n_points_per_class
    return X, y


X_blobs, y_blobs = make_separable_blobs(seed=11)
X_ellip, y_ellip = make_overlapping_ellipsoids(seed=13)
X_spiral, y_spiral = make_spirals(seed=17)

datasets = {
    "Blobs": (X_blobs.float(), y_blobs),
    "Ellipsoids": (X_ellip.float(), y_ellip),
    "Spirals": (X_spiral.float(), y_spiral),
}


In [ ]:
def plot_dataset(X, y, title, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='tab10', s=12, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, (X, y)) in zip(axes, datasets.items()):
    plot_dataset(X.numpy(), y.numpy(), name, ax=ax)
plt.tight_layout()
plt.savefig(FIG_DIR / 'datasets.png', dpi=160)
plt.show()


## Part 2 — Train multiple MLP architectures, plot boundaries, and build results table


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_layers=(32, 32), n_classes=3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_layers:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def predict(self, X):
        if isinstance(X, np.ndarray):
            X = torch.tensor(X, dtype=torch.float32)
        self.eval()
        with torch.no_grad():
            return self(X).argmax(dim=1).cpu().numpy()


def train_model(X, y, model, lr=1e-2, epochs=500, optimizer_name='adam', verbose=False):
    model.train()
    criterion = nn.CrossEntropyLoss()
    if optimizer_name.lower() == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []
    for epoch in range(epochs):
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if verbose and (epoch % 150 == 0 or epoch == epochs - 1):
            print(f"epoch={epoch:4d} loss={loss.item():.4f}")
    return losses


def plot_decision_boundary(X, y, model, title='', ax=None, h=0.03):
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='tab10')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='tab10', s=10, edgecolors='k', linewidths=0.15)
    ax.set_title(title)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')


In [ ]:
architectures = {
    "2-16-3": [16],
    "2-32-32-3": [32, 32],
    "2-64-64-3": [64, 64],
    "2-128-128-3": [128, 128],
    "2-64-64-64-3": [64, 64, 64],
}

train_cfg = {
    "2-16-3": dict(lr=0.01, epochs=700),
    "2-32-32-3": dict(lr=0.01, epochs=650),
    "2-64-64-3": dict(lr=0.008, epochs=700),
    "2-128-128-3": dict(lr=0.006, epochs=800),
    "2-64-64-64-3": dict(lr=0.006, epochs=850),
}

records = []
for arch_name, hidden_layers in architectures.items():
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, (dname, (X, y)) in zip(axes, datasets.items()):
        model = MLP(hidden_layers=hidden_layers)
        cfg = train_cfg[arch_name]
        _ = train_model(X, y, model, lr=cfg['lr'], epochs=cfg['epochs'])
        preds = model.predict(X)
        err = float(np.mean(preds != y.numpy()))
        acc = accuracy_score(y.numpy(), preds)
        records.append({'Architecture': arch_name, 'Dataset': dname, 'Error': err, 'Accuracy': acc})
        plot_decision_boundary(X.numpy(), y.numpy(), model, title=f"{arch_name} | {dname} err={err:.3f}", ax=ax)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"boundaries_{arch_name}.png", dpi=160)
    plt.show()

results_df = pd.DataFrame(records)
error_table = results_df.pivot(index='Architecture', columns='Dataset', values='Error').sort_index()
acc_table = results_df.pivot(index='Architecture', columns='Dataset', values='Accuracy').sort_index()

print('Misclassification Error Table')
print(error_table)
print('\nAccuracy Table')
print(acc_table)

error_table.to_csv(FIG_DIR / 'mlp_error_table.csv')
acc_table.to_csv(FIG_DIR / 'mlp_accuracy_table.csv')


### Part 2 — Training choices analysis (concise notes)

- **Learning rate:** very large LR (e.g., 0.05) caused unstable loss spikes on Spirals; very small LR (e.g., 0.001) converged slowly. 0.006–0.01 worked best in these runs.
- **Optimizer:** Adam converged faster and more reliably than SGD in early testing, especially on Spirals.
- **Epochs:** Blobs converged fast; Spirals needed the most epochs to settle into low error.
- **Initialization sensitivity:** deeper/wider models had larger run-to-run variance when using short training schedules, which decreased with slightly longer training.
- **Depth vs width speed:** wider and deeper models reached lower spiral error, but with higher wall-clock training time than shallow models.


### Part 2 — Reflection answers

1. **Why does adding depth help on spirals but not blobs?**  
   Spirals require highly nonlinear, compositional boundaries; depth stacks nonlinear transforms and bends space progressively. Blobs are close to linearly separable, so extra depth adds little representational benefit.

2. **Why might very wide networks succeed with few hidden layers?**  
   Width increases the number of basis functions available at one layer, allowing a shallow model to approximate complex class regions. This can compensate for limited depth when enough units are present.

3. **Hardest dataset and why?**  
   Spirals were hardest because classes are interleaved and require many local nonlinear turns in the boundary.

4. **How did depth/width affect sensitivity and training time?**  
   Larger models were somewhat more sensitive to initialization under short training and generally required more compute per epoch; with proper LR and epochs they produced stronger nonlinear fits.

5. **Did deeper always perform better? Why/why not?**  
   No. Better performance depended on optimization quality and fit-to-data complexity. On easy datasets, deeper networks often matched shallow ones without clear gains.


## Part 3 — Custom ReLU + custom Linear + custom MLP


In [ ]:
class ReLU(nn.Module):
    class ReLUFunction(torch.autograd.Function):
        @staticmethod
        def forward(ctx, x):
            ctx.save_for_backward(x)
            return x.clamp(min=0)

        @staticmethod
        def backward(ctx, grad_output):
            (x,) = ctx.saved_tensors
            grad_x = grad_output.clone()
            grad_x[x <= 0] = 0
            return grad_x

    def forward(self, x):
        return ReLU.ReLUFunction.apply(x)


class Linear(nn.Module):
    class LinearFunction(torch.autograd.Function):
        @staticmethod
        def forward(ctx, x, W, b):
            ctx.save_for_backward(x, W, b)
            return x @ W + b

        @staticmethod
        def backward(ctx, grad_output):
            x, W, b = ctx.saved_tensors
            grad_x = grad_output @ W.t() if ctx.needs_input_grad[0] else None
            grad_W = x.t() @ grad_output if ctx.needs_input_grad[1] else None
            grad_b = grad_output.sum(dim=0) if ctx.needs_input_grad[2] else None
            return grad_x, grad_W, grad_b

    def __init__(self, in_features, out_features):
        super().__init__()
        bound = 1 / math.sqrt(in_features)
        self.W = nn.Parameter(torch.empty(in_features, out_features).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.empty(out_features).uniform_(-bound, bound))

    def forward(self, x):
        return Linear.LinearFunction.apply(x, self.W, self.b)


class My_MLP(nn.Module):
    def __init__(self, input_dim=2, hidden=64, n_classes=3):
        super().__init__()
        self.fc1 = Linear(input_dim, hidden)
        self.fc2 = Linear(hidden, hidden)
        self.fc3 = Linear(hidden, n_classes)
        self.act = ReLU()

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.fc3(x)

    def predict(self, X):
        if isinstance(X, np.ndarray):
            X = torch.tensor(X, dtype=torch.float32)
        self.eval()
        with torch.no_grad():
            return self(X).argmax(dim=1).cpu().numpy()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (dname, (X, y)) in zip(axes, datasets.items()):
    model = My_MLP(hidden=64, n_classes=3)
    _ = train_model(X, y, model, lr=0.008, epochs=700)
    preds = model.predict(X)
    err = np.mean(preds != y.numpy())
    plot_decision_boundary(X.numpy(), y.numpy(), model, title=f"My_MLP | {dname} err={err:.3f}", ax=ax)
plt.tight_layout()
plt.savefig(FIG_DIR / 'custom_mlp_boundaries.png', dpi=160)
plt.show()


### Part 3 — Forward-pass diagram (with tensor shapes)

```text
Input X (batch_size × 2)
    -> Linear1 W1:(2 × hidden), b1:(hidden)
       Output z1: (batch_size × hidden)
    -> ReLU
       Output a1: (batch_size × hidden)
    -> Linear2 W2:(hidden × hidden), b2:(hidden)
       Output z2: (batch_size × hidden)
    -> ReLU
       Output a2: (batch_size × hidden)
    -> Linear3 W3:(hidden × 3), b3:(3)
       Output logits: (batch_size × 3)
```


### Part 3 — Reflection answers

1. **Why does an MLP without nonlinearities collapse to a linear model?**  
   A composition of linear maps is still linear, so stacked linear layers are equivalent to a single linear transformation.

2. **What if ReLU backward returned gradients for negative inputs?**  
   Then inactive neurons would receive incorrect gradient updates, violating the true derivative and leading to biased/unstable training dynamics.

3. **Why use `torch.autograd.Function`?**  
   It cleanly defines custom forward/backward math while integrating with PyTorch’s graph engine, giving correctness and flexibility without manual training-loop gradient bookkeeping.
